# YOLO11m wood defects - 1024 rect, board-level split
Re-run the training cell after any restart; it resumes from last.pt. Test cell runs ONCE at the very end.

In [ ]:
!pip -q install -U ultralytics
!nvidia-smi --query-gpu=name,memory.total --format=csv
import ultralytics, torch, os, glob, shutil, time
print(ultralytics.__version__, torch.__version__, torch.cuda.device_count(), 'GPUs')
print('inputs:', os.listdir('/kaggle/input'))
SRC = '/kaggle/input/wood-defects-clean/dataset_clean'
assert os.path.isdir(SRC + '/train/images'), os.listdir('/kaggle/input')
hits = glob.glob('/kaggle/input/**/train_yolo11m.py', recursive=True)
assert hits, 'train_yolo11m.py not found under /kaggle/input - is wood-defects-scripts attached?'
SCRIPTS = os.path.dirname(hits[0]); print('scripts dir:', SCRIPTS, os.listdir(SCRIPTS))
# stage on local disk (/kaggle/tmp is not persisted): mounted input reads at ~36 MB/s and is read-only (no label cache)
DATA = '/kaggle/tmp/dataset_clean'
if not os.path.isdir(DATA + '/train/images'):
    t = time.time(); shutil.copytree(SRC, DATA); print(f'copied dataset to local disk in {time.time()-t:.0f}s')
lines = open(DATA + '/data.yaml').read().splitlines()
lines = [('path: ' + DATA) if l.startswith('path:') else l for l in lines]
os.makedirs('/kaggle/working/cfg', exist_ok=True)
with open('/kaggle/working/cfg/data.yaml', 'w') as f:
    f.write('\n'.join(lines) + '\n')
print('\n'.join(lines))

In [ ]:
# restore a previous session's outputs if attached as input (Add Input -> this notebook's output)
import glob, shutil
for prev in glob.glob('/kaggle/input/*/outputs/yolo11m_1024rect'):
    if not os.path.exists('/kaggle/working/outputs/yolo11m_1024rect'):
        shutil.copytree(prev, '/kaggle/working/outputs/yolo11m_1024rect')
        print('restored', prev)

In [ ]:
# Run 2: stability experiment. CFG deviations: batch 32 (was 16), lr0 0.001 (was 0.002), patience 40 (was 25). See NOTES.md section 8.
# Single GPU on purpose: Ultralytics disables rect=True under multi-GPU DDP.
import subprocess, sys
def train(name, batch, lr0=0.001, patience=40):
    cmd = [sys.executable, f'{SCRIPTS}/train_yolo11m.py', '--data', '/kaggle/working/cfg/data.yaml',
           '--project', '/kaggle/working/outputs', '--name', name, '--device', '0',
           '--batch', str(batch), '--lr0', str(lr0), '--patience', str(patience), '--persist', '/kaggle/working/persist']
    print(' '.join(cmd)); rc = subprocess.call(cmd)
    ok = os.path.exists(f'/kaggle/working/outputs/{name}/results.csv')
    print('exit', rc, 'results.csv', ok); return ok
RUN = 'yolo11m_1024rect_b32_lr001'
ok = train(RUN, 32)
if not ok:
    print('batch 32 failed (likely OOM); falling back to batch 24 under a new name')
    RUN = 'yolo11m_1024rect_b24_lr001'
    shutil.rmtree('/kaggle/working/outputs/yolo11m_1024rect_b32_lr001', ignore_errors=True)
    torch.cuda.empty_cache()
    ok = train(RUN, 24)
assert ok, 'training produced no results.csv - see log above'
print(open(f'/kaggle/working/outputs/{RUN}/results.csv').read()[-1500:])

## Test evaluation - run exactly once, after training has finished. Never tune on this.

In [ ]:
#!python {SCRIPTS}/evaluate.py --weights /kaggle/working/outputs/yolo11m_1024rect/weights/best.pt --data /kaggle/working/cfg/data.yaml --split test --device 0 --out /kaggle/working/eval